# Network Collapse Warning — Baseline Simulation

A fixed network handles increasing persistent workload. Overloaded nodes fail and redistribute load, potentially triggering a cascade. This notebook implements the baseline dynamics, **not yet a validated early-warning predictor**.

Run cells from top to bottom. The final cell renders an interactive animation. All numerical settings are provisional.

**Prediction direction:** later estimate a probability distribution over steps remaining until collapse at every settled pre-collapse step. Report a remaining-time estimate, uncertainty intervals, and optional probabilities such as collapse within 5 or 10 steps. These are conditional on the future demand process. Actual remaining time is an outcome (`collapse_step - current_step`), never an input. This notebook generates trajectories; it does not yet fit that predictor.

## Run in Google Colab

Choose **Runtime → Run all**. A CPU runtime is sufficient; no GPU is needed. Colab normally includes NumPy, NetworkX, Matplotlib, and IPython. If an import is missing, run `%pip install numpy networkx matplotlib` in a new Colab cell, then rerun the notebook. No local installation is required.

The HTML animation appears after the plots; use its playback controls. This notebook has been checked for core simulation logic and workload accounting locally; full plot and animation rendering is left for the Colab run.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

SEED = 42
N = 100
MEAN_DEGREE = 6
CAPACITY = 10.0
INITIAL_UTILIZATION = (0.4, 0.7)
DEMAND_INCREMENT = 1.0
COLLAPSE_FRACTION = 0.30
MAX_STEPS = 1000

## Assumptions and update rules

- Simple undirected graph, initially connected; no incoming nodes, rewiring, or recovery.
- Equal fixed capacities and heterogeneous initial loads. Workload accumulates; there is no service/clearing term.
- Each external step injects a fixed **total** demand, divided equally among active nodes.
- Resolve failures synchronously: remove all overloaded nodes in a round before redistributing to neighbors surviving that round. Further overloads fail in the next round.
- A failed node without surviving neighbors contributes its full load to cumulative unserved demand.
- Record both settled states (for later warning evaluation) and within-cascade frames (for animation).
- Stop after a settled step crosses the failure threshold, or at the step limit. Here 30% is an operational definition, not complete loss of function.

Load accounting must hold: active load + unserved demand = initial load + injected demand.

In [ ]:
def resolve_cascade(graph, loads, capacities, active, unserved=0.0):
    """Return a settled copy and a trace of synchronous failure rounds."""
    loads = dict(loads)
    active = set(active)
    trace = []
    while True:
        overloaded = {i for i in active if loads[i] > capacities[i]}
        if not overloaded:
            break
        survivors = active - overloaded
        additions = {i: 0.0 for i in survivors}
        for i in sorted(overloaded):
            recipients = sorted(set(graph.neighbors(i)) & survivors)
            if recipients:
                share = loads[i] / len(recipients)
                for j in recipients:
                    additions[j] += share
            else:
                unserved += loads[i]
            loads[i] = 0.0
        for j, amount in additions.items():
            loads[j] += amount
        active = survivors
        trace.append({'active': set(active), 'loads': dict(loads),
                      'failed': sorted(overloaded), 'unserved': unserved})
    return loads, active, unserved, trace

## Hand-checkable cascade and accounting checks

A, B, C each have capacity 10. Loads 11, 8, 8 cause A to fail first; B and C receive 5.5 each and fail in the next round. Their 27 total load units become unserved demand. Additional checks cover simultaneous failures and the exact-capacity boundary.

In [ ]:
tiny = nx.Graph([('A', 'B'), ('A', 'C')])
loads, active, unserved, trace = resolve_cascade(
    tiny, {'A': 11., 'B': 8., 'C': 8.}, dict.fromkeys(tiny, 10.), set(tiny))
assert [r['failed'] for r in trace] == [['A'], ['B', 'C']]
assert not active and np.isclose(unserved, 27.)
assert np.isclose(trace[0]['loads']['B'], 13.5)
for r in trace:
    print(f"Failed: {r['failed']}; active loads: "
          f"{ {i: r['loads'][i] for i in sorted(r['active'])} }; unserved: {r['unserved']}")

boundary = nx.path_graph(2)
_, alive, lost, rounds = resolve_cascade(boundary, {0: 10., 1: 10.},
                                         {0: 10., 1: 10.}, set(boundary))
assert alive == {0, 1} and not rounds and lost == 0
triangle = nx.complete_graph(3)
l, a, u, r = resolve_cascade(triangle, {0: 11., 1: 11., 2: 0.},
                             dict.fromkeys(triangle, 100.) | {0: 10., 1: 10.}, set(triangle))
assert a == {2} and np.isclose(l[2], 22.) and u == 0
print('Cascade checks passed.')

In [ ]:
def simulate(seed=SEED, n=N, mean_degree=MEAN_DEGREE,
             capacity=CAPACITY, utilization=INITIAL_UTILIZATION,
             delta_d=DEMAND_INCREMENT, collapse_fraction=COLLAPSE_FRACTION,
             max_steps=MAX_STEPS):
    if not (n >= 2 and 0 < mean_degree <= n-1 and capacity > 0
            and 0 <= utilization[0] <= utilization[1] < 1
            and delta_d > 0 and 0 < collapse_fraction <= 1 and max_steps >= 1):
        raise ValueError('Invalid simulation settings')
    rng = np.random.default_rng(seed)
    for attempt in range(10000):
        graph = nx.gnp_random_graph(n, mean_degree/(n-1),
                                   seed=int(rng.integers(0, 2**32-1)))
        if nx.is_connected(graph):
            break
    else:
        raise RuntimeError('Could not sample connected graph; revise connectivity')
    capacities = dict.fromkeys(graph, float(capacity))
    loads = {i: capacity * rng.uniform(*utilization) for i in graph}
    active = set(graph)
    initial = sum(loads.values())
    unserved = injected = 0.0
    history, frames = [], []
    collapse_step = None

    def frame(step, phase, failed=()):
        frames.append(dict(step=step, phase=phase, active=set(active),
                           loads=dict(loads), failed=list(failed)))

    def record(step, cascade_size):
        active_total = sum(loads[i] for i in active)
        assert np.isclose(active_total + unserved, initial + injected)
        spare = sum(capacities[i] - loads[i] for i in active)
        near = sum(loads[i]/capacities[i] >= .9 for i in active)
        giant = max((len(c) for c in nx.connected_components(graph.subgraph(active))), default=0)
        history.append(dict(step=step, injected=injected, failed=n-len(active),
                            failed_fraction=1-len(active)/n, spare=spare/(n*capacity),
                            near_fraction=near/len(active) if active else np.nan,
                            unserved=unserved, active_load=active_total,
                            cascade_size=cascade_size, giant_fraction=giant/n))

    record(0, 0)
    frame(0, 'initial')
    for step in range(1, max_steps+1):
        before = len(active)
        for i in active:
            loads[i] += delta_d / before
        injected += delta_d
        frame(step, 'demand added')
        loads, active, unserved, trace = resolve_cascade(
            graph, loads, capacities, active, unserved)
        for round_number, state in enumerate(trace, 1):
            assert np.isclose(sum(state['loads'][i] for i in state['active'])
                              + state['unserved'], initial + injected)
            frames.append(dict(step=step, phase=f'cascade round {round_number}',
                               active=state['active'], loads=state['loads'],
                               failed=state['failed']))
        record(step, before-len(active))
        if (n-len(active)) >= collapse_fraction*n - 1e-9:
            collapse_step = step
            break
    return dict(graph=graph, capacities=capacities, history=history,
                frames=frames, collapse_step=collapse_step,
                initial_load=initial, attempts=attempt+1)

result = simulate()
repeat = simulate()
assert result['graph'].edges == repeat['graph'].edges
assert result['history'] == repeat['history']
G = result['graph']
print(f'Nodes: {len(G)}, edges: {G.number_of_edges()}, mean degree: {2*G.number_of_edges()/len(G):.2f}')
print(f'Connected graph sampling attempts: {result["attempts"]}')
print(f'Collapse step: {result["collapse_step"]}; final failed nodes: {result["history"][-1]["failed"]}')
print('Workload accounting and seeded reproducibility checks passed.')

## Reading the results

Each horizontal step adds one workload unit. The dashed red line marks observed collapse, not a prediction. Counts and percentages replace fractions:

- **Extra workload added:** excludes the initial load; a straight line is expected under constant demand increments.
- **Failed nodes:** actual count out of the original 100 nodes.
- **Unused capacity:** remaining workload units the active nodes can accept in total. Local connections still constrain redistribution.
- **Nodes close to overload:** percentage of surviving nodes using at least 90% of capacity.

A second graph expands the final cascade into its internal rounds. These rounds happen within one external step, not at separate demand steps.

In [ ]:
h = result['history']
t = np.array([r['step'] for r in h])
collapse = result['collapse_step']
node_count = len(result['graph'])
original_capacity = sum(result['capacities'].values())

with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
    panels = [
        ('Extra workload added', [r['injected'] for r in h], 'Workload units', '#2563eb'),
        ('Failed nodes', [r['failed'] for r in h], f'Number of nodes (out of {node_count})', '#dc2626'),
        ('Unused network capacity', [r['spare']*original_capacity for r in h], 'Available workload units', '#0d9488'),
        ('Nodes close to overload', [100*r['near_fraction'] for r in h], 'Active nodes using ≥90% capacity (%)', '#d97706'),
    ]
    for ax, (title, values, ylabel, color) in zip(axes.flat, panels):
        if title == 'Failed nodes':
            ax.step(t, values, where='post', color=color, linewidth=2.5)
        else:
            ax.plot(t, values, color=color, linewidth=2.5)
        ax.fill_between(t, values, color=color, alpha=.08, step='post' if title=='Failed nodes' else None)
        ax.set_title(title, loc='left', fontweight='bold', pad=12)
        ax.set_ylabel(ylabel)
        ax.set_ylim(bottom=0)
        ax.spines[['top', 'right']].set_visible(False)
        if collapse is not None:
            ax.axvline(collapse, color='#be123c', linestyle='--', alpha=.65,
                       label=f'Observed collapse: step {collapse}')
    axes[0, 1].axhline(COLLAPSE_FRACTION*node_count, color='#64748b', linestyle=':',
                       label=f'Collapse threshold: {COLLAPSE_FRACTION*100:g}%')
    axes[0, 1].set_ylim(0, node_count*1.08)
    axes[0, 1].legend(loc='upper left', fontsize=9)
    axes[1, 1].set_ylim(0, 100)
    for ax in axes[-1]:
        ax.set_xlabel('Simulation step')
    fig.suptitle('How the network approaches collapse', fontsize=17, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, .95))
    plt.show()

# Expand the last observed cascade: an external step can contain many rounds.
cascade_frames = [f for f in result['frames'] if f['failed']]
if cascade_frames:
    last_step = cascade_frames[-1]['step']
    rounds = [f for f in cascade_frames if f['step'] == last_step]
    before = next(f for f in result['frames']
                  if f['step']==last_step and f['phase']=='demand added')
    x = np.arange(len(rounds)+1)
    cumulative = [node_count-len(before['active'])] + [node_count-len(f['active']) for f in rounds]
    newly_failed = [0] + [len(f['failed']) for f in rounds]
    with plt.style.context('seaborn-v0_8-whitegrid'):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].bar(x[1:], newly_failed[1:], color='#f97316')
        axes[0].set_title('New failures in each cascade round', loc='left', fontweight='bold')
        axes[0].set_ylabel('Newly failed nodes')
        axes[1].step(x, cumulative, where='post', color='#dc2626', linewidth=2.5)
        axes[1].scatter(x, cumulative, color='#dc2626', s=25)
        axes[1].axhline(COLLAPSE_FRACTION*node_count, color='#64748b', linestyle=':', label='Collapse threshold')
        axes[1].set_title('Total failures as the cascade spreads', loc='left', fontweight='bold')
        axes[1].set_ylabel('Total failed nodes')
        axes[1].legend()
        for ax in axes:
            ax.set_xlabel('Cascade round (0 = before failures)')
            ax.set_xticks(x)
            ax.set_ylim(bottom=0)
            ax.spines[['top', 'right']].set_visible(False)
        fig.suptitle(f'Inside simulation step {last_step}: the cascade unfolds', fontweight='bold')
        fig.tight_layout()
        plt.show()
else:
    print('No failure cascade occurred in this run.')

## Dynamic network animation

Blue-to-red color shows utilization; gray nodes have failed. Positions stay fixed. We sample quiet steps to keep the animation small but retain every failure round. Use the player controls to pause or step through frames. This is a visualization of one run, not evidence of warning accuracy.

In [ ]:
positions = nx.spring_layout(G, seed=SEED)
all_frames = result['frames']
stride = max(1, len(all_frames)//100)
frames = [f for k, f in enumerate(all_frames)
          if k % stride == 0 or f['failed'] or k == len(all_frames)-1]
fig, ax = plt.subplots(figsize=(8, 6))
norm = Normalize(vmin=0, vmax=1.2, clip=True)
cmap = plt.get_cmap('coolwarm')

def draw(k):
    ax.clear()
    state = frames[k]
    colors = [cmap(norm(state['loads'][i]/result['capacities'][i]))
              if i in state['active'] else (.65, .65, .65, 1) for i in G]
    nx.draw_networkx_edges(G, positions, ax=ax, alpha=.12)
    nx.draw_networkx_nodes(G, positions, ax=ax, node_color=colors, node_size=65)
    ax.set_title(f"Step {state['step']} — {state['phase']} | "
                 f"failed {len(G)-len(state['active'])}/{len(G)}")
    ax.set_axis_off()

fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
             label='Load / capacity (gray = failed)', shrink=.7)
ani = FuncAnimation(fig, draw, frames=len(frames), interval=160)
player = ani.to_jshtml()
plt.close(fig)
display(HTML(player))

## Interpreting the pilot result

With the default seed, this model can jump from no failures to a complete cascade in one external step. Equal capacities, accumulated demand, and conservative redistribution make abrupt failure plausible under these assumptions. Do not interpret a single run as evidence that all real networks behave this way. Capacity heterogeneity, service rates, or recovery are later extensions, not implemented features.

## Can we use differential equations later?

Yes. The baseline is already dynamic, but discrete: demand increments and failure rounds change the state. A continuous-time extension can evolve loads between failures as

$$\frac{dL_i}{dt}=d_i(t)-s_i(L_i)+\kappa\sum_{j\in A(t)} a_{ij}(L_j-L_i),\quad i\in A(t).$$

Here $L_i$ is load, $d_i$ is demand input per unit time, $s_i$ is a service/removal rate, $a_{ij}$ is adjacency between surviving nodes, $\kappa$ controls continuous load balancing, and $A(t)$ is the active node set. The balancing term conserves total active load on an undirected graph. It represents an **additional assumption**, not the baseline's failure-only redistribution.

For a closer continuous-time counterpart of the baseline, use $s_i=0$, $\kappa=0$, and $d_i=D/|A(t)|$ for constant total demand rate $D$. At the event $L_i=C_i$ while crossing upward, fail the node and redistribute its load as an instantaneous jump. Resolve any resulting overload cascade at that same physical time before continuing integration. If several thresholds are reached together, process them synchronously.

This is a **hybrid dynamical system**: differential equations between events plus discrete failure/redistribution rules at events. Differential equations alone do not specify how nodes disappear or transfer their loads. A service term, continuous balancing, recovery, or stochastic demand would change the scientific assumptions and should be introduced one at a time.

## Next experiments

- Inspect several seeds and stress rates before choosing final numerical settings.
- Add small-world and scale-free initial graphs with comparable size and connectivity.
- Add disturbance tests on copies of settled states.
- Calibrate warning rules on training runs; evaluate on independent held-out runs, including a demand-only baseline.
- Compare collapse definitions and report uncertainty. One trajectory cannot establish predictive performance.


The primary evaluation will measure remaining-time estimation error, uncertainty coverage, and probability calibration against elapsed-time/demand baselines. Fixed-horizon alarms are supplementary summaries. Non-collapse runs need censoring-aware evaluation.

## Experiment 2 — Repeat the baseline over 50 random seeds

**Question:** Is abrupt collapse typical under these baseline assumptions, and how much does collapse timing vary?

Keep node count, connectivity target, capacity, initial-load range, demand increment, and collapse definition fixed. Change only the random seed. Each seed regenerates **both the network and initial loads**, so this measures their combined variability; it does not isolate topology from load variation.

Run the baseline definition cells first, then the following two cells. No extra packages are needed. We save summary measurements and settled-state histories as CSV files in the Colab runtime. These are pilot runs, not a training/test evaluation of a predictor.

The plots aligned by actual remaining time use the eventual collapse step **only for retrospective analysis**. A future predictor must not receive that information as an input. Runs without observed collapse are retained and excluded only from plots requiring a known collapse time.

In [ ]:
import csv
from pathlib import Path

EXPERIMENT_SEEDS = range(1000, 1050)
experiment_summaries = []
experiment_histories = {}

for index, seed in enumerate(EXPERIMENT_SEEDS, 1):
    run = simulate(seed=seed)
    history = run['history']
    collapse_step = run['collapse_step']
    first_failure = next((r['step'] for r in history if r['failed'] > 0), None)
    collapsed_same_step = (first_failure == collapse_step) if collapse_step is not None else None
    prior = history[-2] if collapse_step is not None and len(history) >= 2 else None
    experiment_summaries.append({
        'seed': seed,
        'collapse_observed': collapse_step is not None,
        'collapse_step': collapse_step,
        'last_observed_step': history[-1]['step'],
        'first_failure_step': first_failure,
        'steps_from_first_failure_to_collapse': collapse_step-first_failure
            if collapse_step is not None and first_failure is not None else None,
        'collapse_in_first_failure_step': collapsed_same_step,
        'largest_cascade_nodes': max(r['cascade_size'] for r in history),
        'failed_nodes_at_end': history[-1]['failed'],
        'near_overload_percent_one_step_before_collapse':
            100*prior['near_fraction'] if prior is not None else None,
        'actual_mean_degree': 2*run['graph'].number_of_edges()/len(run['graph']),
    })
    # Retain compact histories; do not retain all animation frames across 50 runs.
    experiment_histories[seed] = history
    if index % 10 == 0:
        print(f'Completed {index}/50 runs')

del run
collapsed = [r for r in experiment_summaries if r['collapse_observed']]
collapse_times = np.array([r['collapse_step'] for r in collapsed], dtype=float)
print(f'Observed collapse: {len(collapsed)}/50 runs')
if len(collapse_times):
    print(f'Collapse step: median {np.median(collapse_times):.1f}; '
          f'range {collapse_times.min():.0f}–{collapse_times.max():.0f}')
    print(f'Mean {collapse_times.mean():.1f}; '
          f'sample standard deviation {collapse_times.std(ddof=1):.1f}'
          if len(collapse_times)>1 else 'Only one observed collapse')
    immediate = sum(r['collapse_in_first_failure_step'] for r in collapsed)
    print(f'Collapse in the same external step as first failure: {immediate}/{len(collapsed)} collapse runs')

output_dir = Path('experiment_results')
output_dir.mkdir(exist_ok=True)
with (output_dir/'baseline_50_run_summary.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(experiment_summaries[0]))
    writer.writeheader()
    writer.writerows(experiment_summaries)
with (output_dir/'baseline_50_run_histories.csv').open('w', newline='') as f:
    keys = ['seed', 'collapse_step', 'steps_remaining'] + list(next(iter(experiment_histories.values()))[0])
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    for row in experiment_summaries:
        for state in experiment_histories[row['seed']]:
            writer.writerow({'seed': row['seed'], 'collapse_step': row['collapse_step'],
                             'steps_remaining': row['collapse_step']-state['step']
                                 if row['collapse_observed'] else None, **state})
print('CSV files saved in experiment_results/ (Colab Files panel).')

In [ ]:
with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    if len(collapse_times):
        axes[0].hist(collapse_times, bins='auto', color='#2563eb', edgecolor='white')
        axes[0].axvline(np.median(collapse_times), color='#be123c', linestyle='--', label='Median')
        axes[0].legend()
    else:
        axes[0].text(.5, .5, 'No collapse observed', ha='center', transform=axes[0].transAxes)
    axes[0].set_title('When did collapse happen?', loc='left', fontweight='bold')
    axes[0].set_xlabel('Collapse step')
    axes[0].set_ylabel('Number of runs')

    # A fixed selection of the first 10 seeds, not chosen for visual drama.
    for row in experiment_summaries[:10]:
        history = experiment_histories[row['seed']]
        axes[1].step([r['step'] for r in history], [r['failed'] for r in history],
                     where='post', alpha=.65, label=f"Seed {row['seed']}")
    axes[1].axhline(COLLAPSE_FRACTION*N, color='#64748b', linestyle=':', label='Collapse threshold')
    axes[1].set_title('Gradual damage or sudden failure?', loc='left', fontweight='bold')
    axes[1].set_xlabel('Simulation step')
    axes[1].set_ylabel(f'Total failed nodes (out of {N})')
    axes[1].set_ylim(0, N*1.05)

    # Use only PRE-collapse observations from the final 100 steps.
    aligned = {}
    for row in collapsed:
        for state in experiment_histories[row['seed']]:
            remaining = row['collapse_step']-state['step']
            if 1 <= remaining <= 100 and np.isfinite(state['near_fraction']):
                aligned.setdefault(remaining, []).append(100*state['near_fraction'])
    if aligned:
        remaining = np.array(sorted(aligned))
        median = [np.median(aligned[k]) for k in remaining]
        low = [np.percentile(aligned[k], 10) for k in remaining]
        high = [np.percentile(aligned[k], 90) for k in remaining]
        axes[2].fill_between(remaining, low, high, color='#d97706', alpha=.2,
                             label='10th–90th percentile across available runs')
        axes[2].plot(remaining, median, color='#d97706', linewidth=2.5, label='Median')
        axes[2].invert_xaxis()
        axes[2].legend(fontsize=8)
    axes[2].set_title('Stress before collapse — retrospective', loc='left', fontweight='bold')
    axes[2].set_xlabel('Actual steps remaining until collapse →')
    axes[2].set_ylabel('Active nodes using ≥90% capacity (%)')
    axes[2].set_ylim(0, 100)
    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('50 independent seeded runs — same baseline settings', fontsize=16, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, .94))
    plt.show()

print('Read left to right: variation in lifetime; failure trajectories; stress as collapse approaches.')
print('The shaded band shows variation between runs, not a calibrated prediction interval.')
print('If collapse nearly always follows the first failure in the same step, test capacity heterogeneity next.')